# Train LoRA style Kubo (`kubo_color`) trên Colab T4 miễn phí

Notebook này train một **LoRA style** theo kiểu tranh tô màu tay của Tite Kubo (Bleach): nét mực sạch, đổ bóng bằng marker hoặc màu nước. Bộ ảnh gồm các tranh Orihime, Rangiku, Yoruichi, Soi Fon và Nemu, lấy sẵn từ repo này (`datasets/kubo_style.zip`).

- Model nền: **[WAI-illustrious-SDXL v15](https://civitai.com/models/827184/wai-illustrious-sdxl?modelVersionId=2167369)** (`WAI-Nsfw-Illustrious-15.safetensors`, ~6,9 GB), tải từ Civitai bằng token của bạn.
- Công cụ train: [kohya-ss/sd-scripts](https://github.com/kohya-ss/sd-scripts) bản **v0.12.3** (đã ghim commit), file `sdxl_train_network.py`.
- Từ kích hoạt (trigger): **`kubo_color`**
- Kết quả lưu ở Google Drive: **`MyDrive/LoRA_Kubo`**. Mỗi epoch lưu một file LoRA và một bản trạng thái, nên Colab có ngắt giữa chừng thì chạy lại vẫn **train tiếp từ Drive**.

**Thứ tự:** Runtime → Change runtime type → GPU **T4** → lưu. Sau đó chạy lần lượt **Bước 1 → Bước 6** bằng nút ▶. Cả quá trình mất khoảng **1,5 đến 2 tiếng**, trong đó train khoảng 1 tiếng đến 1 tiếng rưỡi. Để tab Colab mở trong lúc train.

Token Civitai chỉ dán vào **ô mật khẩu ở Bước 2**. Đừng gửi token cho ai và đừng lưu token vào notebook. Cách lấy token: vào [civitai.com/user/account](https://civitai.com/user/account) → **API Keys** → **Add API key**, rồi sao chép token (chỉ hiện một lần).

**Bản sao trên Google Drive không tự cập nhật.** Khi cần bản mới, mở lại [link Colab trên GitHub](https://colab.research.google.com/github/hoangnguyencccc194-blip/rpchat-cd8e0282/blob/main/colab/lora_train_kubo.ipynb).

**Nếu Bước 3 hoặc Bước 5 báo lỗi thư viện:** sd-scripts được kiểm thử chủ yếu trên Python 3.10–3.12, còn Colab hiện mặc định Python 3.13. Khi gặp lỗi, mở Command Palette (Ctrl+Shift+P), chọn **Change runtime version → 2026.07** (Python 3.12), rồi chạy lại từ Bước 1. Bước 4 và Bước 5 có thể chạy lại bao nhiêu lần cũng được.

In [ ]:
#@title Bước 1) Kiểm tra GPU và gắn Google Drive
import subprocess, sys
from pathlib import Path
from IPython.display import display, HTML, Markdown

def _die(msg):
    display(HTML(f'<p style="color:#b00020;font-size:1.05rem"><b>{msg}</b></p>'))
    raise SystemExit(msg)

try:
    import torch
except ImportError:
    _die("Không thấy PyTorch. Hãy dùng runtime Colab mặc định.")

if not torch.cuda.is_available():
    _die("Chưa có GPU. Vào Runtime → Change runtime type → chọn GPU T4 → Save, rồi chạy lại ô này.")

_gpu = torch.cuda.get_device_name(0)
_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · CUDA {torch.version.cuda}")
if _vram < 14:
    _die(f"GPU {_gpu} chỉ có {_vram:.1f} GB VRAM, không đủ để train SDXL. Cần T4 (15 GB) trở lên.")

from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = Path("/content/drive/MyDrive/LoRA_Kubo")
OUT_DIR.mkdir(parents=True, exist_ok=True)
display(Markdown(f"GPU **{_gpu}** ({_vram:.1f} GB). Kết quả sẽ lưu ở `{OUT_DIR}`."))

In [ ]:
#@title Bước 2) Dán token Civitai rồi bấm "Lưu cài đặt"
import ipywidgets as w
from IPython.display import display

# Token chỉ nằm trong bộ nhớ phiên Colab: không ghi ra file, không in ra màn hình.
if "KUBO_CFG" not in globals():
    KUBO_CFG = {}

_token = w.Password(placeholder="Dán API token Civitai vào đây", layout=w.Layout(width="100%"))
_ckpt = w.Dropdown(
    options=[("WAI-illustrious-SDXL v15 (~6,9 GB)", "wai_v15")],
    value="wai_v15",
    description="Model nền:",
    style={"description_width": "110px"},
    layout=w.Layout(width="100%"),
)
_keep = w.Checkbox(
    value=False,
    description="Giữ model nền trên Google Drive (tốn ~6,9 GB Drive, lần sau khỏi tải lại)",
    indent=False,
    layout=w.Layout(width="100%"),
)
_status = w.HTML("")
_btn = w.Button(description="Lưu cài đặt", button_style="success", layout=w.Layout(width="200px", height="40px"))

def _save(_=None):
    tok = (_token.value or "").strip()
    if not tok:
        _status.value = '<p style="color:#b00020"><b>Thiếu token Civitai.</b> Dán token vào ô rồi bấm Lưu cài đặt.</p>'
        return
    KUBO_CFG["token"] = tok
    KUBO_CFG["ckpt_key"] = _ckpt.value
    KUBO_CFG["keep_on_drive"] = bool(_keep.value)
    _token.value = ""
    _status.value = '<p style="color:#1b5e20"><b>Đã lưu.</b> Chạy tiếp Bước 3.</p>'

_btn.on_click(_save)
display(w.VBox([w.HTML("<b>API token Civitai</b>"), _token, _ckpt, _keep, _btn, _status]))

In [ ]:
#@title Bước 3) Cài sd-scripts, tải model nền và bộ ảnh
import os, sys, json, shutil, subprocess, tempfile, urllib.request, urllib.error, zipfile
from pathlib import Path
from IPython.display import display, HTML, Markdown

def _die(msg):
    display(HTML(f'<p style="color:#b00020;font-size:1.05rem"><b>{msg}</b></p>'))
    raise SystemExit(msg)

if "OUT_DIR" not in globals():
    _die("Chưa chạy Bước 1. Chạy Bước 1 trước.")
if "KUBO_CFG" not in globals() or not KUBO_CFG.get("token"):
    _die("Chưa lưu token Civitai. Chạy Bước 2, dán token, bấm Lưu cài đặt, rồi chạy lại ô này.")

ROOT = Path("/content")
SD_DIR = ROOT / "sd-scripts"
SD_COMMIT = "cb70b96858477e716a2096292ab8aab433277f38"  # kohya-ss/sd-scripts v0.12.3 (2026-10-26)
REPO_RAW = "https://raw.githubusercontent.com/hoangnguyencccc194-blip/rpchat-cd8e0282/main"
DATASET_URL = f"{REPO_RAW}/datasets/kubo_style.zip"
DATA_ROOT = ROOT / "lora_kubo" / "dataset"
IMG_DIR = DATA_ROOT / "kubo_style"

CHECKPOINTS = {
    # Đã kiểm tra qua https://civitai.com/api/v1/models/827184 : v15.0, Illustrious, fp16 pruned
    "wai_v15": {
        "version_id": 2167369,
        "filename": "WAI-Nsfw-Illustrious-15.safetensors",
        "bytes": 6_938_040_682,
        "label": "WAI-illustrious-SDXL v15",
    },
}
CKPT = CHECKPOINTS[KUBO_CFG.get("ckpt_key") or "wai_v15"]

def _run(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), **kw)
    if r.returncode != 0:
        _die(f"Lệnh lỗi (mã {r.returncode}). Xem log phía trên.")

# ---- 3a. sd-scripts (ghim commit) + thư viện ----
if not (SD_DIR / ".git").exists():
    _run(["git", "clone", "-q", "https://github.com/kohya-ss/sd-scripts", str(SD_DIR)])
_run(["git", "-C", str(SD_DIR), "fetch", "-q", "origin"])
_run(["git", "-C", str(SD_DIR), "checkout", "-q", SD_COMMIT])

# Giữ nguyên torch/torchvision/numpy có sẵn của Colab để pip không cài đè bản khác.
_freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines()
_keep = [l for l in _freeze if l.split("==")[0].lower() in {"torch", "torchvision", "torchaudio", "numpy"} and "==" in l]
_cons = ROOT / "pip_constraints.txt"
_cons.write_text("\n".join(_keep) + "\n")
print("Giữ nguyên:", ", ".join(_keep))
_run(f"cd {SD_DIR} && {sys.executable} -m pip install -q -c {_cons} -r requirements.txt")
_run(f"{sys.executable} -m pip install -q -c {_cons} timm")

# ---- 3b. Model nền từ Civitai (token gửi qua header, không in ra) ----
MODEL_DIR = (OUT_DIR / "models") if KUBO_CFG.get("keep_on_drive") else (ROOT / "models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / CKPT["filename"]

def _tai_civitai(version_id, dest, token, expected, label):
    if dest.exists() and dest.stat().st_size == expected:
        display(Markdown(f"Đã có **{dest.name}** ({expected/1024**3:.2f} GB), bỏ qua tải."))
        return
    req = urllib.request.Request(f"https://civitai.com/api/v1/model-versions/{version_id}")
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            info = json.load(r)
        if info.get("id") != version_id or info.get("baseModel") != "Illustrious":
            _die(f"Civitai trả về sai phiên bản cho {label}.")
    except urllib.error.URLError:
        print("Không đọc được thông tin phiên bản từ Civitai, vẫn thử tải.")
    part = dest.with_name(dest.name + ".part")
    display(Markdown(f"Đang tải **{label}** (~{expected/1024**3:.1f} GB)… thường mất 3–8 phút."))
    fd, cfg = tempfile.mkstemp(prefix="civitai_", suffix=".cfg")
    try:
        os.chmod(cfg, 0o600)
        with os.fdopen(fd, "w") as f:
            f.write(
                "location\nfail\nsilent\nshow-error\ncontinue-at -\nretry = 5\n"
                f'header = "Authorization: Bearer {token}"\n'
                f'output = "{part}"\n'
                f'url = "https://civitai.com/api/download/models/{version_id}"\n'
            )
        r = subprocess.run(["curl", "--config", cfg], capture_output=True, text=True)
    finally:
        os.remove(cfg)
    size = part.stat().st_size if part.exists() else 0
    if r.returncode != 0 or size != expected:
        if "401" in r.stderr or "403" in r.stderr:
            _die("Civitai từ chối tải (token sai hoặc hết hạn). Tạo token mới, chạy lại Bước 2 rồi Bước 3.")
        _die(f"Tải {label} chưa xong ({size/1024**3:.2f}/{expected/1024**3:.2f} GB). Chạy lại ô này để tải tiếp.")
    part.replace(dest)
    display(Markdown(f"Đã tải xong **{dest.name}**."))

_tai_civitai(CKPT["version_id"], MODEL_PATH, KUBO_CFG["token"], CKPT["bytes"], CKPT["label"])

# ---- 3c. Bộ ảnh từ repo ----
if IMG_DIR.exists():
    shutil.rmtree(IMG_DIR)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
_zip = DATA_ROOT / "kubo_style.zip"
urllib.request.urlretrieve(DATASET_URL, _zip)
with zipfile.ZipFile(_zip) as z:
    z.extractall(DATA_ROOT)
_zip.unlink()
_n = len([p for p in IMG_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}])
if _n == 0:
    _die("Không giải nén được ảnh nào từ datasets/kubo_style.zip.")

KUBO_CFG.update(sd_dir=str(SD_DIR), model_path=str(MODEL_PATH), img_dir=str(IMG_DIR))
display(Markdown(f"Xong Bước 3: sd-scripts `{SD_COMMIT[:7]}`, model `{MODEL_PATH.name}`, **{_n} ảnh** ở `{IMG_DIR}`. Chạy tiếp Bước 4."))

In [ ]:
#@title Bước 4) Tự gắn tag bằng WD14 (thêm trigger kubo_color)
TAGGER_REPO = "SmilingWolf/wd-eva02-large-tagger-v3"  #@param ["SmilingWolf/wd-eva02-large-tagger-v3", "SmilingWolf/wd-vit-tagger-v3"]
NGUONG_TAG = 0.35  #@param {type:"slider", min:0.2, max:0.6, step:0.05}
GAN_LAI_TU_DAU = False  #@param {type:"boolean"}
#@markdown Bỏ chọn `GAN_LAI_TU_DAU` thì caption đã lưu trên Drive (lần chạy trước) được dùng lại, để train tiếp sau khi bị ngắt vẫn đúng caption cũ.

import csv, re, shutil
from pathlib import Path
import torch, timm
from PIL import Image
from huggingface_hub import hf_hub_download
from timm.data import resolve_data_config, create_transform
from IPython.display import display, HTML, Markdown

def _die(msg):
    display(HTML(f'<p style="color:#b00020;font-size:1.05rem"><b>{msg}</b></p>'))
    raise SystemExit(msg)

if "KUBO_CFG" not in globals() or not KUBO_CFG.get("img_dir"):
    _die("Chưa chạy Bước 3. Chạy Bước 1 → 3 trước.")

TRIGGER = "kubo_color"
IMG_DIR = Path(KUBO_CFG["img_dir"])
CAP_BACKUP = OUT_DIR / "captions"
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

# Tên nhân vật cho từng ảnh lấy từ file nhan_vat.txt trong bộ ảnh (dòng dạng "02.jpg: inoue orihime").
# Tagger WD14 không biết Soi Fon và Nemu, và hay nhầm Nemu thành nhân vật khác, nên ưu tiên bảng này.
ORIHIME, RANGIKU, YORUICHI, SOIFON, NEMU = (
    "inoue orihime", "matsumoto rangiku", "shihouin yoruichi", "sui-feng", "kurotsuchi nemu")
NHAN_VAT_THEO_FILE = {}
_bang = IMG_DIR / "nhan_vat.txt"
if _bang.exists():
    for _dong in _bang.read_text(encoding="utf-8").splitlines():
        if ":" in _dong and not _dong.lstrip().startswith("#"):
            _f, _ds = _dong.split(":", 1)
            NHAN_VAT_THEO_FILE[_f.strip()] = [t.strip() for t in _ds.split(";") if t.strip()]
TEN_BLEACH = {ORIHIME, RANGIKU, YORUICHI, SOIFON, NEMU, "kuchiki rukia", "kurosaki ichigo"}

# Tag mô tả nét vẽ và chất liệu: bỏ khỏi caption để LoRA dồn chúng vào kubo_color.
# Tên nhân vật, tóc, trang phục, tư thế, nền... thì giữ lại.
STYLE_TAGS_BO = {
    "traditional media", "copic", "lineart", "sketch", "official art", "anime coloring",
    "flat color", "cel shading", "kubo tite", "tite kubo", "artist name", "scan",
    "scan artifacts", "realistic", "photorealistic", "3d", "anime screencap",
}
STYLE_REGEX_BO = re.compile(r"\(medium\)$")

def _ten(t):
    return t.replace("_", " ").strip()

def _vuong_nen_trang(img):
    img = img.convert("RGBA")
    nen = Image.new("RGBA", img.size, (255, 255, 255, 255))
    nen.alpha_composite(img)
    img = nen.convert("RGB")
    w, h = img.size
    s = max(w, h)
    out = Image.new("RGB", (s, s), (255, 255, 255))
    out.paste(img, ((s - w) // 2, (s - h) // 2))
    return out

def tai_tagger(repo):
    model = timm.create_model("hf-hub:" + repo, pretrained=True).eval()
    names, cats = [], []
    with open(hf_hub_download(repo, "selected_tags.csv"), newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            names.append(_ten(row["name"]))
            cats.append(int(row["category"]))
    cfg = resolve_data_config(model.pretrained_cfg, model=model)
    return model, names, cats, create_transform(**cfg)

def du_doan(model, names, cats, tf, path, device):
    x = tf(_vuong_nen_trang(Image.open(path))).unsqueeze(0)[:, [2, 1, 0]]  # WD v3 nhận BGR
    with torch.inference_mode():
        p = torch.sigmoid(model(x.to(device)).float())[0].cpu().tolist()
    nv = [names[i] for i in sorted(range(len(p)), key=lambda i: -p[i]) if cats[i] == 4 and p[i] >= 0.75]
    chung = [names[i] for i in sorted(range(len(p)), key=lambda i: -p[i]) if cats[i] == 0 and p[i] >= NGUONG_TAG]
    return nv, chung

def lam_caption(path, nv_tagger, chung):
    nv = NHAN_VAT_THEO_FILE.get(path.name)
    if nv is None:
        nv = [t for t in nv_tagger if t in TEN_BLEACH]
    giu = [t for t in chung if t not in STYLE_TAGS_BO and not STYLE_REGEX_BO.search(t)]
    bo = [t for t in chung if t not in giu]
    out = []
    for t in [TRIGGER] + nv + (["bleach"] if nv else []) + giu:
        if t not in out:
            out.append(t)
    return ", ".join(out), bo

def chay_gan_tag():
    anh = sorted(p for p in IMG_DIR.iterdir() if p.suffix.lower() in IMG_EXT)
    if not anh:
        _die(f"Không thấy ảnh trong {IMG_DIR}. Chạy lại Bước 3.")
    if GAN_LAI_TU_DAU and CAP_BACKUP.exists():
        shutil.rmtree(CAP_BACKUP)
    can_tag = []
    for p in anh:
        bk = CAP_BACKUP / (p.stem + ".txt")
        if bk.exists():
            shutil.copy(bk, p.with_suffix(".txt"))
        else:
            can_tag.append(p)
    da_bo = {}
    if can_tag:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Đang tải tagger {TAGGER_REPO} …")
        model, names, cats, tf = tai_tagger(TAGGER_REPO)
        model.to(device)
        for i, p in enumerate(can_tag, 1):
            nv, chung = du_doan(model, names, cats, tf, p, device)
            cap, bo = lam_caption(p, nv, chung)
            for t in bo:
                da_bo[t] = da_bo.get(t, 0) + 1
            p.with_suffix(".txt").write_text(cap + "\n", encoding="utf-8")
            print(f"\r  đã gắn tag {i}/{len(can_tag)}", end="")
        print()
        del model
        if device == "cuda":
            torch.cuda.empty_cache()
        CAP_BACKUP.mkdir(parents=True, exist_ok=True)
        for p in can_tag:
            shutil.copy(p.with_suffix(".txt"), CAP_BACKUP / (p.stem + ".txt"))
    else:
        print(f"Dùng lại {len(anh)} caption đã lưu ở {CAP_BACKUP}.")
    return anh, da_bo

_anh, _da_bo = chay_gan_tag()
thieu = [p.name for p in _anh if not p.with_suffix(".txt").read_text(encoding="utf-8").startswith(TRIGGER)]
if thieu:
    _die(f"Caption thiếu trigger {TRIGGER}: {thieu}. Bật GAN_LAI_TU_DAU rồi chạy lại.")
KUBO_CFG["n_img"] = len(_anh)
display(Markdown(f"Đã có caption cho **{len(_anh)} ảnh**, mỗi caption bắt đầu bằng `{TRIGGER}`. "
                 f"Tag style đã bỏ: {', '.join(f'{k} ×{v}' for k, v in sorted(_da_bo.items())) or 'không có'}."))
for p in _anh[::8][:5]:
    print(f"{p.name}: {p.with_suffix('.txt').read_text(encoding='utf-8').strip()}\n")

In [ ]:
#@title Bước 5) Train LoRA (tự train tiếp nếu Drive đã có trạng thái)
SO_BUOC_MUC_TIEU = 1800  #@param {type:"slider", min:1200, max:2400, step:100}
SO_EPOCH = 10  #@param {type:"integer"}
OPTIMIZER = "AdamW8bit"  #@param ["AdamW8bit", "Prodigy", "AdamW"]
LEARNING_RATE = 1e-4  #@param {type:"number"}
NETWORK_DIM = 16  #@param {type:"integer"}
NETWORK_ALPHA = 8  #@param {type:"integer"}
TRAIN_TIEP_TU_DRIVE = True  #@param {type:"boolean"}
#@markdown - Số lần lặp ảnh (repeats) tự tính từ số ảnh để tổng số bước gần `SO_BUOC_MUC_TIEU`.
#@markdown - Mỗi epoch lưu `kubo_color-0000NN.safetensors` và thư mục `…-state` vào `MyDrive/LoRA_Kubo`. Nếu Colab ngắt: chạy lại Bước 1 → 5, notebook tự tìm trạng thái mới nhất và train tiếp.
#@markdown - Prodigy tự chỉnh learning rate (đặt sẵn 1.0, bỏ qua ô `LEARNING_RATE`).

import os, re, subprocess, sys
from pathlib import Path
from IPython.display import display, HTML, Markdown

def _die(msg):
    display(HTML(f'<p style="color:#b00020;font-size:1.05rem"><b>{msg}</b></p>'))
    raise SystemExit(msg)

if "KUBO_CFG" not in globals() or not KUBO_CFG.get("n_img"):
    _die("Chưa gắn tag. Chạy Bước 1 → 4 trước.")

SD_DIR = Path(KUBO_CFG["sd_dir"])
MODEL_PATH = Path(KUBO_CFG["model_path"])
IMG_DIR = Path(KUBO_CFG["img_dir"])
N_IMG = int(KUBO_CFG["n_img"])
OUTPUT_NAME = "kubo_color"
CFG_DIR = Path("/content/lora_kubo/config")
CFG_DIR.mkdir(parents=True, exist_ok=True)

REPEATS = max(1, round(SO_BUOC_MUC_TIEU / (N_IMG * SO_EPOCH)))
STEPS_PER_EPOCH = N_IMG * REPEATS  # batch 1
TOTAL_STEPS = STEPS_PER_EPOCH * SO_EPOCH
WARMUP = int(TOTAL_STEPS * 0.05)

dataset_toml = f"""[general]
enable_bucket = true
caption_extension = ".txt"
shuffle_caption = false
keep_tokens = 1

[[datasets]]
resolution = 1024
batch_size = 1
min_bucket_reso = 512
max_bucket_reso = 2048
bucket_reso_steps = 64
bucket_no_upscale = false

  [[datasets.subsets]]
  image_dir = "{IMG_DIR}"
  num_repeats = {REPEATS}
"""
(CFG_DIR / "dataset.toml").write_text(dataset_toml)

if OPTIMIZER == "Prodigy":
    opt_args = ["--optimizer_type", "Prodigy", "--learning_rate", "1.0", "--unet_lr", "1.0",
                "--optimizer_args", "decouple=True", "weight_decay=0.01", "use_bias_correction=True",
                "safeguard_warmup=True", "d_coef=1.0", "--lr_scheduler", "cosine"]
else:
    opt_args = ["--optimizer_type", OPTIMIZER, "--learning_rate", str(LEARNING_RATE), "--unet_lr", str(LEARNING_RATE),
                "--optimizer_args", "weight_decay=0.1", "--lr_scheduler", "cosine", "--lr_warmup_steps", str(WARMUP)]

def _state_moi_nhat():
    states = []
    for d in OUT_DIR.glob(f"{OUTPUT_NAME}-*-state"):
        m = re.fullmatch(rf"{re.escape(OUTPUT_NAME)}-(\d+)-state", d.name)
        if m and d.is_dir() and (d / "train_state.json").exists():
            states.append((int(m.group(1)), d))
    return max(states)[1] if states else None

resume = _state_moi_nhat() if TRAIN_TIEP_TU_DRIVE else None
if TRAIN_TIEP_TU_DRIVE and (OUT_DIR / f"{OUTPUT_NAME}.safetensors").exists():
    _die(f"Drive đã có bản cuối {OUTPUT_NAME}.safetensors, tức là đã train xong. Chạy Bước 6 để xem file, "
         "hoặc bỏ chọn TRAIN_TIEP_TU_DRIVE để train lại từ đầu (file cũ sẽ bị ghi đè).")

cmd = [
    sys.executable, "-m", "accelerate.commands.launch",
    "--num_processes", "1", "--num_machines", "1", "--mixed_precision", "fp16",
    "--dynamo_backend", "no", "--num_cpu_threads_per_process", "2",
    "sdxl_train_network.py",
    "--pretrained_model_name_or_path", str(MODEL_PATH),
    "--dataset_config", str(CFG_DIR / "dataset.toml"),
    "--output_dir", str(OUT_DIR), "--output_name", OUTPUT_NAME,
    "--save_model_as", "safetensors", "--save_precision", "fp16",
    "--network_module", "networks.lora",
    "--network_dim", str(NETWORK_DIM), "--network_alpha", str(NETWORK_ALPHA),
    "--network_train_unet_only",
    *opt_args,
    "--max_train_epochs", str(SO_EPOCH),
    "--mixed_precision", "fp16", "--gradient_checkpointing", "--sdpa",
    "--cache_latents", "--cache_latents_to_disk",
    "--cache_text_encoder_outputs", "--cache_text_encoder_outputs_to_disk",
    "--no_half_vae", "--lowram", "--max_token_length", "225",
    "--min_snr_gamma", "5", "--seed", "42",
    "--max_data_loader_n_workers", "1", "--persistent_data_loader_workers",
    "--save_every_n_epochs", "1", "--save_state", "--save_last_n_epochs_state", "2",
    "--logging_dir", str(OUT_DIR / "logs"),
]
if resume is not None:
    cmd += ["--resume", str(resume)]

display(Markdown(
    f"**{N_IMG} ảnh × {REPEATS} lần lặp × {SO_EPOCH} epoch = {TOTAL_STEPS} bước** (batch 1). "
    f"Optimizer {OPTIMIZER}, dim {NETWORK_DIM} / alpha {NETWORK_ALPHA}, 1024 px có bucket. "
    + (f"Train tiếp từ `{resume.name}`." if resume is not None else "Train từ đầu.")
    + " Trên T4 mỗi bước khoảng 2–3 giây; vài phút đầu dành cho nạp model và cache latent."))

proc = subprocess.Popen(cmd, cwd=str(SD_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    _die("Train dừng vì lỗi (xem log phía trên). Nếu hết bộ nhớ GPU: giảm NETWORK_DIM hoặc chạy lại; "
         "nếu Colab bị ngắt: chạy lại Bước 1 → 5 để train tiếp từ Drive.")
display(Markdown("**Train xong.** Chạy Bước 6 để xem file LoRA."))

In [ ]:
#@title Bước 6) Xem file LoRA trên Drive
from pathlib import Path
from IPython.display import display, Markdown

OUT_DIR = Path("/content/drive/MyDrive/LoRA_Kubo")
files = sorted(OUT_DIR.glob("*.safetensors"), key=lambda p: p.stat().st_mtime)
if not files:
    print("Chưa có file LoRA nào trong", OUT_DIR)
for p in files:
    print(f"{p.stat().st_size/1024/1024:7.1f} MB  {p.name}")
if files:
    display(Markdown(
        f"File mới nhất: **`{files[-1].name}`** (file không có số epoch là bản cuối cùng). "
        "Nên thử vài epoch cuối, ví dụ 8, 9 và 10, vì bản cuối chưa chắc đẹp nhất."))

## Dùng LoRA trong notebook ComfyUI Bleach

1. Mở [notebook ComfyUI Bleach](https://colab.research.google.com/github/hoangnguyencccc194-blip/rpchat-cd8e0282/blob/main/colab/comfyui_bleach.ipynb), chạy ô 1 và ô 2 như bình thường.
2. Chép file LoRA vào thư mục loras của ComfyUI. Ví dụ, chạy trong một ô mới của notebook ComfyUI (sau khi gắn Drive):
   ```
   !cp "/content/drive/MyDrive/LoRA_Kubo/kubo_color.safetensors" /content/ComfyUI/models/loras/
   ```
   Hoặc dán link file vào ô "Link LoRA thêm" nếu bạn đưa file lên Civitai.
3. Trong ComfyUI, thêm node **LoraLoader**, chọn `kubo_color.safetensors`, **weight 0.7–0.9** (bắt đầu với 0.8).
4. Prompt bắt đầu bằng từ kích hoạt **`kubo_color`**, rồi tên nhân vật và bối cảnh, ví dụ:
   `kubo_color, inoue orihime, bleach, 1girl, solo, smile, white background, masterpiece, best quality`
5. Nếu ảnh bị cứng hoặc lặp lại ảnh mẫu, hạ weight xuống 0.6 hoặc dùng file epoch sớm hơn. Có thể tắt LoRA Bleach Watercolor và Tite Kubo Style để thấy rõ style mới.